# SHAP Explainability for GoEmotions RoBERTa
Fixed version — corrects path resolution, empty-set handling, and SHAP batching.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip -q install shap transformers torch pandas numpy matplotlib seaborn scikit-learn


In [3]:
import json
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import shap
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import confusion_matrix


## ✅ Fix 1: Correct Path Resolution
Original code had a broken `find_repo_root()` that always checked a hardcoded absolute path
instead of traversing the directory tree. We now directly set paths from the known Drive mount point.

In [4]:
# ── FIXED: Direct path configuration ─────────────────────────────────────────
# Set DRIVE_PROJECT_ROOT to wherever your project lives in Google Drive.
# Adjust this one line if your folder name differs.
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/Emotional Aware Chatbot')

EXP_ROOT   = DRIVE_PROJECT_ROOT / 'experiments' / 'goemotions_roberta_full_pipeline'
DATA_DIR   = EXP_ROOT / 'data' / 'processed'
REPORTS_DIR = EXP_ROOT / 'reports'
SHAP_DIR   = REPORTS_DIR / 'shap'
SHAP_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_MODEL_DIR = EXP_ROOT / 'models' / 'best_model'
CANDIDATE_CHECKPOINT_DIRS = [
    EXP_ROOT / 'results' / 'trainer_runs',
    DRIVE_PROJECT_ROOT / 'results' / 'roberta_runs',
]

def has_model_files(model_dir: Path) -> bool:
    return (
        model_dir.exists()
        and (model_dir / 'config.json').exists()
        and (
            (model_dir / 'model.safetensors').exists()
            or (model_dir / 'pytorch_model.bin').exists()
        )
    )

def find_latest_checkpoint(base_dir: Path):
    if not base_dir.exists():
        return None
    checkpoints = [p for p in base_dir.glob('checkpoint-*') if p.is_dir() and has_model_files(p)]
    if not checkpoints:
        return None
    return max(checkpoints, key=lambda p: int(re.search(r'checkpoint-(\d+)', p.name).group(1)))

if has_model_files(PRIMARY_MODEL_DIR):
    MODEL_DIR = PRIMARY_MODEL_DIR
    model_source = 'best_model'
else:
    fallback = None
    for base in CANDIDATE_CHECKPOINT_DIRS:
        ckpt = find_latest_checkpoint(base)
        if ckpt is not None:
            fallback = ckpt
            break
    if fallback is None:
        raise FileNotFoundError(
            f'No trained model found. Searched:\n'
            f'  {PRIMARY_MODEL_DIR}\n'
            + '\n'.join(f'  {d}/checkpoint-*' for d in CANDIDATE_CHECKPOINT_DIRS)
            + '\nRun training notebook first.'
        )
    MODEL_DIR = fallback
    model_source = 'checkpoint_fallback'

TRAIN_PATH = DATA_DIR / 'train.csv'
VALID_PATH = DATA_DIR / 'valid.csv'
TEST_PATH  = DATA_DIR / 'test.csv'
for p in [TRAIN_PATH, VALID_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'Missing processed split: {p}')

print('Project root :', DRIVE_PROJECT_ROOT)
print('Experiment   :', EXP_ROOT)
print('Model source :', model_source)
print('Model path   :', MODEL_DIR)
print('SHAP out dir :', SHAP_DIR)


Project root : /content/drive/MyDrive/Emotional Aware Chatbot
Experiment   : /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline
Model source : best_model
Model path   : /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/models/best_model
SHAP out dir : /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/reports/shap


In [5]:
# Runtime controls
SHAP_BACKGROUND_SIZE  = 100
SHAP_EXPLAIN_SIZE     = 200
SHAP_FOCUS_NEUTRAL_ERRORS = 60
MAX_LENGTH  = 128
BATCH_SIZE  = 64
SHAP_BATCH  = 8    # smaller batches prevent OOM during SHAP perturbations
SEED        = 42

np.random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', device)


Device: cuda


In [6]:
# Load data and model
train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH)
test_df  = pd.read_csv(TEST_PATH)

model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()

id2label = model.config.id2label
if isinstance(id2label, dict):
    labels = [id2label[i] for i in range(len(id2label))]
else:
    labels = sorted(test_df['final_emotion'].unique().tolist())

label2id = {l: i for i, l in enumerate(labels)}

print('Labels:', labels)
print('Train/Valid/Test:', train_df.shape, valid_df.shape, test_df.shape)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Labels: ['Anger', 'Fear', 'Joy', 'Love', 'Neutral', 'Sadness']
Train/Valid/Test: (95078, 2) (11885, 2) (11885, 2)


In [7]:
def predict_proba(texts):
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()
    texts = [str(t) for t in texts]

    probs_all = []
    with torch.no_grad():
        for i in range(0, len(texts), BATCH_SIZE):
            batch = texts[i:i+BATCH_SIZE]
            enc = tokenizer(
                batch,
                truncation=True,
                max_length=MAX_LENGTH,
                padding=True,
                return_tensors='pt',
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            logits = model(**enc).logits
            probs  = torch.softmax(logits, dim=-1).detach().cpu().numpy()
            probs_all.append(probs)

    return np.vstack(probs_all)


In [8]:
# Build confusion map to focus on weak boundaries
sample_for_confusion = test_df.sample(n=min(1200, len(test_df)), random_state=SEED).copy().reset_index(drop=True)
probs_cf    = predict_proba(sample_for_confusion['text'].tolist())
pred_ids_cf = probs_cf.argmax(axis=1)
sample_for_confusion['pred_label'] = [labels[i] for i in pred_ids_cf]

cm = confusion_matrix(
    sample_for_confusion['final_emotion'],
    sample_for_confusion['pred_label'],
    labels=labels,
)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.to_csv(SHAP_DIR / 'confusion_matrix_sample.csv')

neutral_errors = sample_for_confusion[
    (sample_for_confusion['final_emotion'] == 'Neutral') &
    (sample_for_confusion['pred_label']   != 'Neutral')
].copy()

print('Confusion matrix saved to:', SHAP_DIR / 'confusion_matrix_sample.csv')
print('Neutral->non-neutral errors in sample:', len(neutral_errors))


Confusion matrix saved to: /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/reports/shap/confusion_matrix_sample.csv
Neutral->non-neutral errors in sample: 75


## ✅ Fix 2: Guard Against Empty `neutral_errors`
If your model rarely misclassifies Neutral (good sign at 85% accuracy!),
`neutral_errors` can be empty, causing SHAP to receive 0 samples and crash.
We now fall back gracefully to random test samples.

In [9]:
# Select texts for SHAP explanations
background_texts = train_df['text'].sample(
    n=min(SHAP_BACKGROUND_SIZE, len(train_df)), random_state=SEED
).astype(str).tolist()

# ── FIXED: guard against empty neutral_errors ─────────────────────────────────
if len(neutral_errors) > 0:
    focus_texts = neutral_errors['text'].head(SHAP_FOCUS_NEUTRAL_ERRORS).astype(str).tolist()
    print(f'Using {len(focus_texts)} Neutral misclassification examples as focus.')
else:
    focus_texts = []
    print('No Neutral errors found (model is doing well!). Using random test samples only.')

remaining_n  = max(0, SHAP_EXPLAIN_SIZE - len(focus_texts))
random_pool  = test_df[~test_df['text'].isin(focus_texts)]
random_texts = random_pool['text'].sample(
    n=min(remaining_n, len(random_pool)), random_state=SEED
).astype(str).tolist()

explain_texts = focus_texts + random_texts

if len(explain_texts) == 0:
    raise ValueError('explain_texts is empty — check your test_df and text column name.')

print('Background size:', len(background_texts))
print('Explain size   :', len(explain_texts))


Using 60 Neutral misclassification examples as focus.
Background size: 100
Explain size   : 200


## ✅ Fix 3: SHAP with `batch_size` to Prevent OOM / Timeout
The original code called `explainer(explain_texts)` with no batch size,
which tries to perturb all 200 samples at once and often crashes on Colab.
We add `batch_size=SHAP_BATCH` (default 8) and a progress bar.

In [10]:
masker   = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(predict_proba, masker, output_names=labels)

# batch_size controls how many texts are perturbed per forward pass group
# Lower = safer for memory, higher = faster. 8 is stable on Colab T4.
print(f'Running SHAP on {len(explain_texts)} samples (batch_size={SHAP_BATCH})...')
shap_values = explainer(explain_texts, batch_size=SHAP_BATCH)
print('SHAP values shape:', shap_values.values.shape)


Running SHAP on 200 samples (batch_size=8)...


PartitionExplainer explainer:  76%|███████▌  | 151/200 [02:26<01:16,  1.56s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 201it [03:11,  1.04it/s]

SHAP values shape: (200,)


In [12]:
# Build per-example local explanations table
probs    = predict_proba(explain_texts)
pred_ids = probs.argmax(axis=1)

truth_lookup = dict(zip(test_df['text'].astype(str), test_df['final_emotion']))
local_rows   = []

for i, txt in enumerate(explain_texts):
    pred_idx   = int(pred_ids[i])
    pred_label = labels[pred_idx]
    true_label = truth_lookup.get(txt, 'NA')

    # Fix: Correctly index the shap_values.values array.
    # shap_values.values is a 1D array of 2D arrays, so we first get the 2D array for sample `i`,
    # then index that 2D array to get token values for the predicted class.
    token_vals = shap_values.values[i][:, pred_idx]
    token_data = shap_values.data[i]

    token_pairs = []
    for tok, val in zip(token_data, token_vals):
        tok = str(tok)
        if tok.strip() == '' or tok in {'<s>', '</s>', '<pad>'}:
            continue
        token_pairs.append((tok, float(val)))

    token_pairs_sorted = sorted(token_pairs, key=lambda x: x[1], reverse=True)
    top_pos = token_pairs_sorted[:8]
    top_neg = sorted(token_pairs, key=lambda x: x[1])[:8]

    local_rows.append({
        'text'               : txt,
        'true_label'         : true_label,
        'pred_label'         : pred_label,
        'pred_confidence'    : float(probs[i, pred_idx]),
        'top_positive_tokens': ' | '.join([f"{t}:{v:.4f}" for t, v in top_pos]),
        'top_negative_tokens': ' | '.join([f"{t}:{v:.4f}" for t, v in top_neg]),
    })

local_df = pd.DataFrame(local_rows)
local_df.to_csv(SHAP_DIR / 'local_explanations.csv', index=False)
print('Saved:', SHAP_DIR / 'local_explanations.csv')
local_df.head(5)

Saved: /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/reports/shap/local_explanations.csv


,text,true_label,pred_label,pred_confidence,top_positive_tokens,top_negative_tokens
0,"Sorry, no trivia today. Calendar had a fun fac...",Neutral,Joy,0.974713,fun :0.6895 | trivia :0.1986 | . :0.1200 | a :...,"Sorry:-0.1932 | , :-0.1463 | edit :-0.0828 | p..."
1,They got bored from haunting earth for thousan...,Neutral,Anger,0.992085,bored :0.3313 | got :0.2040 | and :0.1304 | fr...,for :-0.0371 | moved :-0.0287 | years :-0.0234...
2,r/woooosh I guess... xD,Neutral,Joy,0.797736,... :0.4174 | x:0.2469 | D:0.1760 | ooo:0.1572...,w:-0.1574 | osh :-0.0856 | guess:-0.0508 | /:-...
3,Probably the thief trashed the place,Neutral,Anger,0.998814,ashed :0.5352 | tr:0.4596 | thief :0.2084 | pl...,the :-0.2048 | the :-0.0183 | Probably :-0.004...
4,*Hey just noticed..* it's your **5th Cakeday**...,Neutral,Love,0.982391,your :0.2031 | just :0.1675 | ay:0.1054 | th :...,'s :-0.0526 | (:-0.0337 | ):-0.0268 | ** :-0.0...


In [14]:
# Global token importance per class
class_token_scores = {lbl: defaultdict(float) for lbl in labels}
class_token_counts = {lbl: defaultdict(int)   for lbl in labels}

# Fix: shap_values.values is a 1D array of 2D arrays (tokens x classes).
# Get num_samples from its length and num_classes from the shape of the first sample's values.
num_samples = len(shap_values.values)
# Ensure there's at least one sample to get num_classes
num_classes = shap_values.values[0].shape[1] if num_samples > 0 else 0

for i in range(num_samples):
    token_data = [str(t) for t in shap_values.data[i]]
    for t_idx, tok in enumerate(token_data):
        tok = tok.strip()
        if tok == '' or tok in {'<s>', '</s>', '<pad>'}:
            continue
        for c_idx, lbl in enumerate(labels):
            # Fix: Access the 2D array for the current sample first, then index by token and class.
            v = float(abs(shap_values.values[i][t_idx, c_idx]))
            class_token_scores[lbl][tok] += v
            class_token_counts[lbl][tok] += 1

rows = []
for lbl in labels:
    for tok, total in class_token_scores[lbl].items():
        mean_abs = total / max(class_token_counts[lbl][tok], 1)
        rows.append({'label': lbl, 'token': tok, 'mean_abs_shap': mean_abs})

global_df = pd.DataFrame(rows).sort_values(['label', 'mean_abs_shap'], ascending=[True, False])
global_df.to_csv(SHAP_DIR / 'global_token_importance.csv', index=False)
print('Saved:', SHAP_DIR / 'global_token_importance.csv')

topk = 20
for lbl in labels:
    top_df = global_df[global_df['label'] == lbl].head(topk).iloc[::-1]
    plt.figure(figsize=(8, 6))
    plt.barh(top_df['token'], top_df['mean_abs_shap'])
    plt.title(f'Top {topk} SHAP Tokens — {lbl}')
    plt.xlabel('Mean |SHAP|')
    plt.tight_layout()
    out_path = SHAP_DIR / f'global_top_tokens_{lbl}.png'
    plt.savefig(out_path, dpi=160)
    plt.close()

print('Saved per-class top-token plots in:', SHAP_DIR)

Saved: /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/reports/shap/global_token_importance.csv
Saved per-class top-token plots in: /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/reports/shap


In [15]:
# Validation + index file
expected_files = [
    SHAP_DIR / 'confusion_matrix_sample.csv',
    SHAP_DIR / 'local_explanations.csv',
    SHAP_DIR / 'global_token_importance.csv',
]
for p in expected_files:
    assert p.exists() and p.stat().st_size > 0, f'Missing or empty file: {p}'

png_files = sorted(SHAP_DIR.glob('global_top_tokens_*.png'))
assert len(png_files) >= 1, 'No global SHAP plots found.'

index = {
    'model_path'     : str(MODEL_DIR),
    'model_source'   : model_source,
    'labels'         : labels,
    'background_size': SHAP_BACKGROUND_SIZE,
    'explain_size'   : len(explain_texts),
    'files'          : [str(p) for p in expected_files] + [str(p) for p in png_files],
}
with open(SHAP_DIR / 'shap_report_index.json', 'w') as f:
    json.dump(index, f, indent=2)

print('✅ SHAP pipeline complete!')
print('Index saved to:', SHAP_DIR / 'shap_report_index.json')


✅ SHAP pipeline complete!
Index saved to: /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/reports/shap/shap_report_index.json
